<a href="https://colab.research.google.com/github/takao-takenouchi/k-anonymization_sql/blob/main/k_anonymization_sql.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 初めに
このノートブックでは、
個人情報データベースのサンプルデータを用いて、「k-匿名化」をSQLで行います。
「k-匿名化」の処理の流れを、簡単に理解することを目的にしています。

## 匿名化処理をSQLで体験

### Step1 テーブルの準備

In [1]:
# DuckDBを使ってSQLで処理します
!pip -q install duckdb

サンプル用のファイルをダウンロードして使います。（ファイルをアップロードすることも可能です）

In [2]:
import urllib.request

url = "https://raw.githubusercontent.com/takao-takenouchi/k-anonymization_sql/main/anonymization_table.csv"
urllib.request.urlretrieve(url, "anonymization_table.csv")

## ファイルをアップロードしたい場合は、以下のコードをつかいます。「ファイル選択」をクリックして、「anonymization_table.csv」をアップロードしてください。
# from google.colab import files
# uploaded = files.upload()

('anonymization_table.csv', <http.client.HTTPMessage at 0x7bfee2f26510>)

アップロードしたCSVファイルを、Databaseに読み込んで、SQLのテーブル作成します

In [3]:
import duckdb

con = duckdb.connect()

# 文字列型(VARVHAR型)で読み込みます。ファイル名は先ほどアップロードした"anonymization_table.csv"です
con.execute("""
CREATE OR REPLACE TABLE anonymization AS
SELECT
    CAST(column0 AS VARCHAR) AS gender,
    CAST(column1 AS VARCHAR) AS post,
    CAST(column2 AS VARCHAR) AS age,
    CAST(column3 AS VARCHAR) AS disease
FROM read_csv_auto(
    'anonymization_table.csv',
    header = false
);
""")



# 一応テーブルの詳細を表示します
con.execute("DESCRIBE anonymization").df()

,column_name,column_type,null,key,default,extra
0,gender,VARCHAR,YES,None,None,None
1,post,VARCHAR,YES,None,None,None
2,age,VARCHAR,YES,None,None,None
3,disease,VARCHAR,YES,None,None,None


In [4]:
# 作成したテーブルも表示します。5000行のファイルのはずです。
con.execute("SELECT * FROM anonymization").df()

,gender,post,age,disease
0,M,881-0006,28,162
1,M,699-1234,53,937
2,M,684-0024,44,076
3,M,500-8126,43,372
4,M,900-0031,48,781
...,...,...,...,...
4995,W,251-0043,49,105
4996,W,310-0818,26,324
4997,W,511-0284,26,146
4998,W,026-0051,47,951


### Step2 匿名化前のkの値をチェック



まずは、匿名化前のテーブルのkの値をチェックします。
準識別子でGroup byしてCountを取っていくだけです。簡単です。

In [8]:
con.execute("""
    SELECT gender, post, age, COUNT(*) AS k
    FROM anonymization
    GROUP BY gender, post, age
""").df()

,gender,post,age,k
0,M,881-0006,28,1
1,M,900-0031,48,1
2,M,869-2506,54,1
3,W,895-1804,59,1
4,W,916-0206,51,1
...,...,...,...,...
4987,W,849-0304,43,1
4988,M,878-0206,51,1
4989,W,939-1426,58,1
4990,M,406-0805,45,1


In [5]:
con.execute("""
SELECT
    k,
    COUNT(*) AS group_count
FROM (
    SELECT gender, post, age, COUNT(*) AS k
    FROM anonymization
    GROUP BY gender, post, age
) t
GROUP BY k
ORDER BY k
""").df()

,k,group_count
0,1,4984
1,2,8


### Step3 匿名化処理


続いて匿名化します。ここでは、Global Recodingで、一部の桁を削除する方法で行います。

In [6]:
con.execute("""
CREATE OR REPLACE TABLE anon_1 AS
SELECT
    gender,
    SUBSTRING(post, 1, 1) || '**-****' AS post,
    SUBSTRING(age, 1, 1) || '*' AS age,
    disease
FROM anonymization
""")

con.execute("SELECT * FROM anon_1").df()

,gender,post,age,disease
0,M,8**-****,2*,162
1,M,6**-****,5*,937
2,M,6**-****,4*,076
3,M,5**-****,4*,372
4,M,9**-****,4*,781
...,...,...,...,...
4995,W,2**-****,4*,105
4996,W,3**-****,2*,324
4997,W,5**-****,2*,146
4998,W,0**-****,4*,951


### Step4 匿名化後のkの値をチェック

匿名化後のテーブルのkの値をチェックします。同じく、Group byしてCountを取っていくだけです。

In [7]:
con.execute("""
SELECT
    k,
    COUNT(*) AS group_count
FROM (
    SELECT gender, post, age, COUNT(*) AS k
    FROM anon_1
    GROUP BY gender, post, age
) t
GROUP BY k
ORDER BY k
""").df()

,k,group_count
0,5,1
1,6,1
2,13,1
3,14,1
4,16,3
5,19,1
6,20,1
7,22,2
8,27,1
9,30,3


## まとめ

- SQLで匿名化処理の概要を体験
- Global Recodingで桁の削除であれば、SQL一発
- 匿名化前と匿名化後のkの値を比較も簡単。kの値のチェックはGroup byしてCountを取っていくだけ
